In [1503]:
import math as m
import numpy as np
import random 
import stable_baselines3
#import gym 
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import random 

from abc import ABC, abstractmethod



In [1504]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
#optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


## Building the Env

In [1505]:
# redefine box and env  as in box is singular isolated , part of a bigger formula , pattern , structure made by the bigger class .
#test the stuff here 

class CellBox :
    def __init__(
            self,
            name,
            xy:tuple[int,int],
            color:str, 
            symbol:str=None, 
            dirlist : list = None, 
            reward_fn = None
        ):

        # this part describes the box itself
        self.name=name
        self.coordinate=xy
        self.color=color
        self.reward=0
        
        self.reward_fn = reward_fn if reward_fn != None else self.default_rewardfn
        
        self.symbol= symbol
        #this one is about the surrounding of the box
        self.neighbours={
            "up":dirlist[0],
            "down":dirlist[1],
            "right":dirlist[2],
            "left":dirlist[3],
            "stand":self
        }

    def default_rewardfn(self):
        if self.symbol == "X":
            self.reward = -100
        elif self.symbol == "O":
            self.reward = 100
        else:
            self.reward = -10

        return self.reward

    def reward_(self):
        return  self.reward_fn()

    def next_state(self,action):
        if self.neighbours[action] is None:
            return self
        return self.neighbours[action] 
    
    # 🔥 KEY FIX: equality based on identity of state meaning
    def __eq__(self, other):
        return isinstance(other, CellBox) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [1506]:
class Board(ABC): # i could ask for completion and correction on this board , but i think it is not important
                   # basically not a priority
    def __init__(self,
                 cells: list[CellBox],
                 geometry="box",
                 data=None):

        self.cells = cells
        self.geometry = geometry
        self.data = data

        self.form()

    def form(self):

        if self.geometry == "box":
            self.make_box(self.data)

        elif self.geometry == "pyramid":
            self.make_pyramid(self.data)

        elif self.geometry == "stack":
            self.make_stack(self.data)

        elif self.geometry == "h_line":
            self.make_line({"direction": "h"})

        elif self.geometry == "v_line":
            self.make_line({"direction": "v"})

        elif self.geometry == "custom":
            self.make_custom(self.data)

    # -------------------------------------------------------

    def __iter__(self):
        return iter(self.cells)

    def __len__(self):
        return len(self.cells)

    def snapshot(self):
        return self.cells.copy()

    # -------------------------------------------------------

    def make_stack(self, data):
        pass

    def make_box(self, data):

        shape = data["shape"]
        N, M = shape

        cells = self.cells

        for m in range(M):
            for n in range(N):

                p = n + m * N
                s = cells[p]

                s.neighbours = {
                    "up": None,
                    "down": None,
                    "right": None,
                    "left": None,
                    "stand": s
                }

                if n < N - 1:
                    s.neighbours["right"] = cells[p + 1]

                if n > 0:
                    s.neighbours["left"] = cells[p - 1]

                if m < M - 1:
                    s.neighbours["up"] = cells[p + N]

                if m > 0:
                    s.neighbours["down"] = cells[p - N]

    def make_pyramid(self, data):
        pass

    def make_custom(self, data): 
        
        opposite = {
            "right": "left",
            "left":  "right",
            "up":    "down",
            "down":  "up",
            }
        # Reset every cell first
        # for cell in self.cells:

        #     cell.neighbours = {
        #         "up": None,
        #         "down": None,
        #         "left": None,
        #         "right": None,
        #         "stand": cell,
        #     }
        #         "right": None,
        #         "stand": cell,
        #     }

        # Apply all user-defined connections
        for cell_a, direction, cell_b in data["connections"]:

            if direction not in opposite:
                raise ValueError(
                    f"Unknown direction '{direction}'. "
                    f"Allowed: {list(opposite.keys())}"
                )

            # Forward connection
            cell_a.neighbours[direction] = cell_b

            # Reverse connection
            cell_b.neighbours[opposite[direction]] = cell_a

    def make_line(self, data):

        direction = data["direction"]

        if direction == "h":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["right"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["left"] = self.cells[i]

        elif direction == "v":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["up"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["down"] = self.cells[i]

    

In [1507]:
# Env (organizing the puzzle of state into env )
class Env:

    @property
    def actions(self) -> list:
        """All actions available in this environment."""
        raise NotImplementedError

    @property
    def states(self) -> list:
        """All states in this environment."""
        raise NotImplementedError

    def reset(self):
        """Start a new episode. Return the initial state."""
        raise NotImplementedError

    def step(self, action):
        """
        Apply action to the current state.
        Return (next_state, reward, done).
        """
        raise NotImplementedError
    

In [1508]:
class Chessboard(Env):
    """
    RL environment built on top of a Board.

    Board
        -> geometry

    Environment
        -> transitions
        -> rewards
        -> terminal conditions
        -> current state
    """

    def __init__(self,
                 board: Board,
                 actions: list[str],
                 start: CellBox):

        self.board = board

        self._actions = actions

        self.start = start
        self.current = start
        # insert self.path = [] saving all the exps that the agent went through

    # -------------------------------------------------------

    @property
    def states(self):
        return self.board.cells

    @property
    def actions(self):
        return self._actions

    # -------------------------------------------------------

    def reset(self):

        self.current = self.start
        return self.current

    # -------------------------------------------------------

    def step(self, action):

        next_state = self.current.next_state(action)

        next_state.reward_()

        reward = float( next_state.reward )

        done = next_state.symbol in ("X", "O")

        self.current = next_state

        return {
            "state": self.current,
            "action": action,
            "reward": reward,
            "next_state": next_state,
            "done": done,
        }

    # -------------------------------------------------------

    def snapshot(self):

        return {
            "current": self.current,
            "start": self.start,
            "board": self.board.snapshot(),
        }

### Setting an example board

In [1509]:
#set up
s1 = CellBox("s1",(0,0),"white","X",[None,None,None,None])
s2 = CellBox("s2",(2,0),"white","O",[None,None,None,None])
s3 = CellBox("s3",(4,0),"white","X",[None,None,None,None])
s4 = CellBox("s4",(0,1),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"grey",None,[None,None,None,None])
s6 = CellBox("s6",(2,1),"white",None,[None,None,None,None])
s7 = CellBox("s7",(3,1),"grey",None,[None,None,None,None])
s8 = CellBox("s8",(4,1),"white",None,[None,None,None,None])

#s0 = CellBox("empty box")

#define
s1.neighbours = {"up": s4,   "down": None, "right": None, "left": None, "stand": s1}
s2.neighbours = {"up": s6,   "down": None, "right": None, "left": None, "stand": s2}
s3.neighbours = {"up": s8,   "down": None, "right": None, "left": None, "stand": s3}
s4.neighbours = {"up": None, "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": None, "down": None, "right": s6,   "left": s4,   "stand": s5}
s6.neighbours = {"up": None, "down": s2,   "right": s7,   "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": None, "right": s8,   "left": s6,   "stand": s7}
s8.neighbours = {"up": None, "down": s3,   "right": None, "left": s7,   "stand": s8}
#fwefsdf
States1=[s1,s2,s3,s4,s5,s6,s7,s8]

Actions=["up", "right","down","left"]




In [1510]:
s1 = CellBox("s1",(2,0),"white",None,[None,None,None,None])
s2 = CellBox("s2",(2,1),"white",None,[None,None,None,None])
s3 = CellBox("s3",(2,2),"white",None,[None,None,None,None])
s4 = CellBox("s4",(1,0),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"white","X",[None,None,None,None])
s6 = CellBox("s6",(1,2),"white",None,[None,None,None,None])
s7 = CellBox("s7",(0,0),"white",None,[None,None,None,None])
s8 = CellBox("s8",(0,1),"white",None,[None,None,None,None])
s9 = CellBox("s9",(0,2),"white","O",[None,None,None,None])

States2 = [s1,s2,s3,s4,s5,s6,s7,s8,s9]

claude_board = Board(States2,geometry="box",data={"shape":(3,3)})

env = Chessboard(
    board=claude_board,
    actions=Actions,
    start=s1
)

state = env.reset()

experience = env.step("right")

In [1511]:
experience["next_state"].name

isinstance(env, Env)

True

## Define the State vector

In [1512]:
# #reorganize the whole code here

# def manhattan_goal(state:CellBox):
#         x,y = state.coordinate
#         return abs(x-0) + abs(y-2)

# def manhattan_trap(state:CellBox):
#         x,y = state.coordinate
#         return abs(x-1) + abs(y-1)


# #the state _ vector
# def state_to_vector(state):

#     color = 0 if state.color == "white" else 1

#     if state.symbol == "X":
#         symbol = -1
#     elif state.symbol == "O":
#         symbol = 1
#     else:
#         symbol = 0

#     return np.array([
#         state.coordinate[0],
#         state.coordinate[1],
#         color,
#         symbol,
#         manhattan_goal(state=state),
#         manhattan_trap(state=state)
#     ], dtype=np.float32)

In [1513]:
def state_to_vector(state):
    # encoding: -1=trap, 0=empty, 1=goal, 2=wall
    obstacles = {"up": 0.0, "right": 0.0, "down": 0.0, "left": 0.0, "self": 0.0}

    if state.symbol == "X":   obstacles["self"] = -1.0
    elif state.symbol == "O": obstacles["self"] =  1.0

    for direction in ["up", "right", "down", "left"]:
        neighbour = state.neighbours[direction]
        if neighbour is None:
            obstacles[direction] = 2.0           # wall
        elif neighbour.symbol == "X":
            obstacles[direction] = -1.0          # trap
        elif neighbour.symbol == "O":
            obstacles[direction] =  1.0          # goal
        # else stays 0.0 = empty or self

    return np.array([
        obstacles["up"],
        obstacles["right"],
        obstacles["down"],
        obstacles["left"],
        obstacles["self"]
    ], dtype=np.float32)

## Policy

In [1514]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

### Some examples

In [1515]:
class DeterministicPolicy(Policy):
    def __init__(self):
        self.mapping = {}
    def select_action(self, state):        return self.mapping[state]
    def action_distribution(self, state):  return {self.mapping[state]: 1.0}
    def snapshot(self):                    return self.mapping.copy()

class StochasticPolicy(Policy):
    def __init__(self):
        self.pi = {}
    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]
    def action_distribution(self, state):  return self.pi[state]
    def snapshot(self):                    return self.pi.copy()

class Policy_theta_softmax(Policy):
    def __init__(self, phi, theta): # phi:FeatureFunction
        self.pi      = {}
        self.weights = theta
        self.phi     = phi

    def _zeller(self, state, action, Actions): #
        n  = Actions.index(action)
        a1 = Actions[(n + 1) % 4]
        a2 = Actions[(n + 3) % 4]
        phi_sa = 0.8 * self.phi.compute(state.next_state(action)) + \
                 0.1 * (self.phi.compute(state.next_state(a1)) +
                        self.phi.compute(state.next_state(a2)))
        h = (self.weights @ phi_sa).item()
        return np.exp(h)

    def calculate_all_probabilities(self, state, Actions):
        # compute each zeller once, reuse for denominator
        zellers = {a: self._zeller(state, a, Actions) for a in Actions}
        N = sum(zellers.values())
        self.pi[state] = {a: float(z / N) for a, z in zellers.items()}

    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]

    def action_distribution(self, state):  return self.pi.get(state, {})
    def snapshot(self):                    return self.pi.copy()

class NeuralPolicy(Policy):
    def __init__(self, network):
        self.network = network
    def select_action(self, state):
        probs = self.network(state)
        return 0  # placeholder
    def action_distribution(self, state):  return self.network(state)
    def snapshot(self):                    return self.network.state_dict()

## Learning Strategy (for improvement and update)

In [1516]:
from abc import ABC, abstractmethod

class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

class QLearning(LearningStrategy):
    def __init__(self, actions, alpha=0.1, gamma=0.9):
        self.alpha   = alpha
        self.gamma   = gamma
        self.actions = actions
        self.Q       = {}

    def _ensure(self, s):
        if s not in self.Q:
            self.Q[s] = {a: 0.0 for a in self.actions}

    def update(self, policy, exp):
        s, a, r, next_s, done = exp["state"], exp["action"], exp["reward"], exp["next_state"], exp["done"]
        self._ensure(s)
        self._ensure(next_s)
        target = r if done else r + self.gamma * max(self.Q[next_s].values())
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])
        policy.mapping[s] = max(self.Q[s], key=self.Q[s].get)

    def snapshot(self):
        return self.Q.copy()

class Reinforce(LearningStrategy):
    def __init__(self, lr=0.1):
        self.lr = lr
    def update(self, policy, state_vec, action_index, reward):
        probs = policy.forward(state_vec)
        for i in range(policy.W.shape[1]):
            policy.W[0, i] += self.lr * reward * ((1 if i == action_index else 0) - probs[i]) * state_vec[0]
    def snapshot(self): return {}



## Generalized Advantage Estimation

In [1517]:
def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages


### example


In [1518]:
# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 0.43):
   A[ 1] = 0.430
--------------------------------------------------
After step 2 (new δ = -0.646):
   A[ 1] = -0.178
   A[ 2] = -0.646
--------------------------------------------------
After step 3 (new δ = 1.543):
   A[ 1] = 1.187
   A[ 2] = 0.805
   A[ 3] = 1.543
--------------------------------------------------
After step 4 (new δ = 1.876):
   A[ 1] = 2.748
   A[ 2] = 2.465
   A[ 3] = 3.307
   A[ 4] = 1.876
--------------------------------------------------
After step 5 (new δ = 1.902):
   A[ 1] = 4.236
   A[ 2] = 4.047
   A[ 3] = 4.990
   A[ 4] = 3.665
   A[ 5] = 1.902
--------------------------------------------------
After step 6 (new δ = 0.499):
   A[ 1] = 4.603
   A[ 2] = 4.437
   A[ 3] = 5.405
   A[ 4] = 4.106
   A[ 5] = 2.371
   A[ 6] = 0.499
--------------------------------------------------


## Blocks to build Neural Network

In [1519]:
# Uncomment if you want to use this simple version Brick 1 and 2 

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → activation"""
    def __init__(self, dim, activation=nn.ReLU):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act  = activation()

    def forward(self, x):
        return self.act(self.norm(self.fc(x)))

# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 64,
        n_layers: int = 2,
        activation=nn.ReLU
    ):
        super().__init__()

        self.input_layer = nn.Linear(n_features, hidden_dim)

        layers = [
            self.input_layer,
            activation()
        ]

        for i in range(1, n_layers + 1):
            setattr(self, f"block{i}", Block(hidden_dim, activation))

        for i in range(1, n_layers + 1):
            layers.append(getattr(self, f"block{i}"))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# class SharedBody(nn.Module): # efficient version
#     def __init__(
#         self,
#         n_layers: int,
#         n_features: int,
#         hidden_dim: int = 64,
#         activation=nn.ReLU
#     ):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(n_features, hidden_dim),
#             activation(),
#             *[Block(hidden_dim, activation) for _ in range(n_layers)]
#         )

        
#vars() was useful here 

In [1520]:
# # Uncomment if you want to use this simple version Brick 1 and 2 

#── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
# class Block(nn.Module):
#     """A reusable residual-style chunk: linear → norm → relu"""
#     def __init__(self, dim):
#         super().__init__()
#         self.fc   = nn.Linear(dim, dim)
#         self.norm = nn.LayerNorm(dim)

#     def forward(self, x):
#         return F.relu(self.norm(self.fc(x)))  # ← fixed: F.relu() not nn.ReLU()


#── BRICK 2: shared body ─────────────────────────────────────────────────
# class SharedBody(nn.Module):
#     """
#     Encodes the state φ(s) into a shared representation.
#     Both actor and critic read from this — they see the same features.
#     Input:  φ(s) of shape (n_features,)
#     Output: hidden representation of shape (hidden_dim,)
#     """
#     def __init__(self, n_features: int, hidden_dim: int = 64):
#         super().__init__()
#         self.input_layer = nn.Linear(n_features, hidden_dim)
#         self.block1      = Block(hidden_dim)
#         self.block2      = Block(hidden_dim)

#         #self.network = nn.Sequential(....)

#     def forward(self, x):
#         x = F.relu(self.input_layer(x))
#         x = self.block1(x)
#         x = self.block2(x)
#         return x

In [1521]:

# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, n_actions: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, n_actions)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (n_actions,)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, n_features: int, n_actions: int, hidden_dim: int = 64,n_layers = 2):
        super().__init__()
        self.body   = SharedBody(n_features, hidden_dim,n_layers)
        self.actor  = ActorHead(hidden_dim, n_actions)
        self.critic = CriticHead(hidden_dim)

    def forward(self, x):
        h     = self.body(x)
        probs = self.actor(h)    # π_θ(a|s)
        value = self.critic(h)   # V_θ(s)
        return probs, value

    def get_action(self, state_vec: np.ndarray):
        """
        Given a raw state vector, returns:
          action   — sampled from π_θ(a|s)
          log_prob — log π_θ(a|s), needed for PPO ratio
          value    — V_θ(s), stored in rollout buffer
        """
        x           = torch.tensor(state_vec, dtype=torch.float32)
        probs, value = self.forward(x)
        dist        = torch.distributions.Categorical(probs)
        action_idx  = dist.sample()
        log_prob    = dist.log_prob(action_idx)
        return action_idx.item(), log_prob, value


# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    

In [1522]:
class Policy_PPO(Policy):
    def __init__(self,network,actions,state_to_vector_fn = state_to_vector):
        super().__init__()
        self.pi ={}
        self.net = network
        self.actions = actions
        self.state_to_vector_fn = state_to_vector_fn
        #self.log = []
        #self.value = {} # to store the values for each state - defined externally
        # not the primary fn of pi but we can save some computation energy
         
    def select_action(self, state):
        x = self.state_to_vector_fn(state)
        a_idx , log_prob , value = self.net.get_action(x) # see what we can do with the value
        #self.log.append(log_prob)
        #self.value[state] = value
        return self.actions[a_idx] , log_prob, value

    def action_distribution(self, state):
        x = torch.tensor(self.state_to_vector_fn(state),dtype= torch.float32)
        with torch.no_grad() :
            probs,_ = self.net(x)
        
        return probs
    

    def snapshot(self): return self.net.state_dict()
        

### Examples for sanity check

In [1523]:
# ── quick sanity check ───────────────────────────────────────────────────
n_features = 2   # [dist_goal, dist_trap] — your phi(s) size
n_actions  = 4   # up, right, down, left

net = ActorCritic(n_features=n_features, n_actions=n_actions, hidden_dim=64, n_layers= 100)

dummy_state = torch.tensor([2.0, 1.0], dtype=torch.float32)
probs, value = net(dummy_state)

print(f"Action probs : {probs.detach().numpy()}")  # 4 numbers summing to 1
print(f"Value        : {value.item():.4f}")        # single scalar
print(f"Total params : {sum(p.numel() for p in net.parameters())}")

# act from a real numpy feature vector
state_vec = np.array([2.0, 1.0])
action_idx, log_prob, val = net.get_action(state_vec) 
print(f"Sampled action index: {action_idx} → {Actions[action_idx]}")

Action probs : [0.48843583 0.22770138 0.09528968 0.18857306]
Value        : -0.8729
Total params : 429317
Sampled action index: 0 → up


In [1524]:
#vars(net.body)

### example to see if everything works together

In [1525]:
example_state = env.states[0]
n_features = len(state_to_vector(example_state))
pi = Policy_PPO(ActorCritic(n_features= n_features , n_actions=n_actions, hidden_dim=64),Actions) 
pi.select_action(s6) #Actions[a_idx] , log_prob, value

('up',
 tensor(-0.9248, grad_fn=<SqueezeBackward1>),
 tensor(0.3110, grad_fn=<SqueezeBackward1>))

## Defining Loss class for Improvement / Learning / Updating

In [1526]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError



## My Loss function for PPO

In [1527]:
class Loss_Entropy(Loss):
    """L_H = -Σ p·log(p)  — encourages exploration"""
    def forward(self, probs: torch.Tensor) -> torch.Tensor:
        eps = 1e-8
        return -(probs * torch.log(probs + eps)).sum(dim=-1).mean()

class Loss_Critic(Loss):
    """L_V = (V(s) - y)²  — makes critic predict returns accurately"""
    def forward(self, v_pred: torch.Tensor, v_target: torch.Tensor) -> torch.Tensor:
        #return ((v_pred - v_target) ** 2).mean()
        return F.mse_loss(v_pred, v_target)
    
class Loss_Actor(Loss):
    """L_pi = -ρ·A  — PPO clipped policy gradient"""
    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(self, log_prob_new: torch.Tensor,
                      log_prob_old: torch.Tensor,
                      advantage:   torch.Tensor) -> torch.Tensor:
        # ratio ρ = π_new / π_old  via log difference (numerically stable)
        rho     = torch.exp(log_prob_new - log_prob_old)
        clipped = torch.clamp(rho, 1 - self.eps, 1 + self.eps)
        # take the pessimistic (minimum) of clipped and unclipped
        L_clip  = torch.min(rho * advantage, clipped * advantage)
        return -L_clip.mean()   # negative because we MAXIMIZE, but optimizer MINIMIZES

class Loss_PPO(Loss):
    """Total PPO loss = L_actor + cv·L_critic - ce·L_entropy"""
    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()
        self.cv    = cv
        self.ce    = ce
        self.L_pi  = Loss_Actor(clip_epsilon)
        self.L_v   = Loss_Critic()
        self.L_h   = Loss_Entropy()

    def forward(self,
                log_prob_new: torch.Tensor,
                log_prob_old: torch.Tensor,
                advantage:    torch.Tensor,
                v_pred:       torch.Tensor,
                v_target:     torch.Tensor,
                probs:        torch.Tensor) -> torch.Tensor:

        actor_loss  = self.L_pi(log_prob_new, log_prob_old, advantage)
        critic_loss = self.L_v(v_pred, v_target)
        entropy     = self.L_h(probs)

        return actor_loss + self.cv * critic_loss - self.ce * entropy

In [1528]:

class PPO(LearningStrategy):
    def __init__(self,
                optimizer,
                clip_epsilon: float = 0.2,
                cv:           float = 0.5,    # critic loss weight
                ce:           float = 0.01,   # entropy bonus weight
                gamma:        float = 0.99,
                lam:          float = 0.95,
                n_epochs:     int   = 1):
        self.optimizer    = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv           = cv       
        self.ce           = ce          
        self.gamma        = gamma
        self.lam          = lam
        self.n_epochs     = n_epochs
        self.loss_fn      = Loss_PPO(cv=cv, ce=ce,
                                    clip_epsilon=clip_epsilon)

    def compute_gae(self, deltas: list, # warning : this methode is only with ppo used 
                    dones:list, # otherwise delete this one 
                    gamma: float = None,
                    lam:   float = None) -> torch.Tensor:
        gamma = gamma if gamma is not None else self.gamma
        lam   = lam   if lam   is not None else self.lam
        T     = len(deltas)
        advantages = torch.zeros(T)
        gae        = torch.tensor(0.0)
        for t in reversed(range(T)):
            if dones[t]:
                gae = torch.tensor(0.0)
            gae           = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy: Policy_PPO, rollout: dict) -> float:

        deltas         = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old  = rollout["log_probs_old"]
        actions_taken  = rollout["actions_taken"]
        values_visited = rollout["values_visited"]
        dones = rollout["dones"] # added to fix the leaking

        A        = self.compute_gae(deltas, dones) # added dones to fix the leaking
        v_tensor = torch.stack([v.detach() for v in values_visited])
        returns  = (A + v_tensor).detach()

        #normalize advantages — stabilizes training
        if torch.isfinite(A).all() and A.std() > 1e-8: #I added the torch.isfinite to solve the NaN issue 
            A = (A - A.mean()) / (A.std() + 1e-8)

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new, values_new, probs_all = [], [], []

            for s, a in zip(states_visited, actions_taken):
                x        = torch.as_tensor(policy.state_to_vector_fn(s), dtype=torch.float32)
                probs, v = policy.net(x)
                dist     = torch.distributions.Categorical(probs)
                log_p    = dist.log_prob(torch.tensor(policy.actions.index(a)))
                log_probs_new.append(log_p)
                values_new.append(v)
                probs_all.append(probs)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack(log_probs_old)
            v_pred        = torch.stack(values_new)
            probs_tensor  = torch.stack(probs_all)

            loss = self.loss_fn(log_probs_new, log_probs_old_t,
                                A, v_pred, returns, probs_tensor)

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            self.optimizer.step()
            total_loss += loss.item()

        return total_loss / self.n_epochs

    def snapshot(self) -> dict:
        return {
            "gamma":        self.gamma,
            "lam":          self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv":           self.cv,
            "ce":           self.ce,
            "lr":           self.optimizer.param_groups[0]["lr"],
    }

In [1529]:
def debug_ppo(env, agent, rollout=None):
    """
    rollout is optional.
    Pass the rollout dict before agent.learn(...) if you want to inspect
    deltas, advantages and returns.
    """

    print("\n" + "="*70)
    print("PPO COMPLETE DEBUG")
    print("="*70)

    ############################################################
    # 1. Environment
    ############################################################

    print("\n[1] ENVIRONMENT")
    print("-"*70)

    for s in env.states:
        for action in agent.policy.actions:

            old = env.current
            env.current = s

            exp = env.step(action)


            env.current = old

            ns = exp["next_state"]

            print(
                f"{s.name:>3} -- {action:<6}"
                f" --> {ns.name:<3}"
                f" reward={exp['reward']:>6}"
                f" done={exp['done']}"
                f" symbol={ns.symbol}"
            )

    ############################################################
    # 2. Critic
    ############################################################

    print("\n" + "="*70)
    print("[2] VALUE FUNCTION")
    print("="*70)

    for s in env.states:

        x = torch.as_tensor(agent.policy.state_to_vector_fn(s), dtype=torch.float32)

        with torch.no_grad():
            _, v = agent.policy.net(x)

        print(
            f"{s.name:<3}"
            f" symbol={str(s.symbol):<5}"
            f" value={v.item():10.3f}"
        )

    ############################################################
    # 3. Actor
    ############################################################

    print("\n" + "="*70)
    print("[3] POLICY")
    print("="*70)

    for s in env.states:

        x = torch.as_tensor(agent.policy.state_to_vector_fn(s), dtype=torch.float32)

        with torch.no_grad():
            probs, _ = agent.policy.net(x)

        probs = probs.tolist()

        print(f"\n{s.name}")

        for a, p in zip(agent.policy.actions, probs):
            print(f"   {a:<6}: {p:.4f}")

        best = probs.index(max(probs))
        print(f"   best action : {agent.policy.actions[best]}")

    ############################################################
    # 4. Rollout inspection
    ############################################################

    if rollout is not None:

        print("\n" + "="*70)
        print("[4] ROLLOUT")
        print("="*70)

        A = agent.strategy.compute_gae(
            rollout["deltas"],
            rollout["dones"]
        )

        values = torch.stack(rollout["values_visited"])
        returns = A + values

        for i in range(len(rollout["states_visited"])):

            state = rollout["states_visited"][i]

            print(
                f"\nstep {i:02d}"
                f"  state={state.name}"
                f"  action={rollout['actions_taken'][i]}"
                f"  done={rollout['dones'][i]}"
            )

            print(f"   value      : {values[i].item():10.3f}")
            print(f"   delta      : {rollout['deltas'][i].item():10.3f}")
            print(f"   advantage  : {A[i].item():10.3f}")
            print(f"   return     : {returns[i].item():10.3f}")

    ############################################################
    # 5. Parameters
    ############################################################

    print("\n" + "="*70)
    print("[5] PARAMETERS")
    print("="*70)

    for name, p in agent.policy.net.named_parameters():

        print(f"\n{name}")
        print(f" mean : {p.data.mean().item():.6f}")
        print(f" std  : {p.data.std().item():.6f}")
        print(f" min  : {p.data.min().item():.6f}")
        print(f" max  : {p.data.max().item():.6f}")
        print(f" nan  : {torch.isnan(p.data).any().item()}")
        print(f" inf  : {torch.isinf(p.data).any().item()}")

    ############################################################
    # 6. Gradients
    ############################################################

    print("\n" + "="*70)
    print("[6] GRADIENTS")
    print("="*70)

    for name, p in agent.policy.net.named_parameters():

        if p.grad is None:
            continue

        print(f"\n{name}")
        print(f" norm : {p.grad.norm().item():.6f}")
        print(f" mean : {p.grad.mean().item():.6f}")
        print(f" std  : {p.grad.std().item():.6f}")
        print(f" nan  : {torch.isnan(p.grad).any().item()}")
        print(f" inf  : {torch.isinf(p.grad).any().item()}")

    print("\n" + "="*70)
    print("END DEBUG")
    print("="*70)

In [1530]:

def debug_learning_balance(policy):
    """
    Compare how strongly the actor and critic are learning.
    Call AFTER loss.backward() and BEFORE optimizer.step().
    """

    actor_sq = 0.0
    critic_sq = 0.0

    for name, param in policy.net.named_parameters():

        if param.grad is None:
            continue

        g = param.grad.detach().norm().item()

        if "actor" in name:
            actor_sq += g ** 2

        elif "critic" in name:
            critic_sq += g ** 2

    actor_norm = actor_sq ** 0.5
    critic_norm = critic_sq ** 0.5

    ratio = actor_norm / (critic_norm + 1e-8)

    print("=" * 60)
    print("LEARNING BALANCE")
    print("=" * 60)
    print(f"Actor gradient : {actor_norm:.6f}")
    print(f"Critic gradient: {critic_norm:.6f}")
    print(f"Ratio A/C      : {ratio:.3f}")

    if 0.7 <= ratio <= 1.3:
        print("Status : Balanced")

    elif ratio < 0.7:
        print("Status : Critic dominates")

    else:
        print("Status : Actor dominates")

    return {
        "actor": actor_norm,
        "critic": critic_norm,
        "ratio": ratio
    }

## Create the Agent

In [1531]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        

        self.strategy.update(self.policy, experience)
        self.memory.append((
            experience["state"].name, 
            experience["action"], 
            experience["reward"], 
            experience["next_state"].name, 
            experience["done"]
        ))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [1532]:

class Agent_PPO(Agent):

    def __init__(self, strategy: PPO, policy: Policy_PPO):
        super().__init__(strategy, policy)

    def act(self, state):
        return self.policy.select_action(state)

    def learn(self, rollout):
        loss = self.strategy.update(self.policy, rollout)
        self.memory.append({
            "rollout_size": len(rollout["deltas"]), 
            "loss":loss
        })

    def value(self, state):
        x = torch.as_tensor(
            self.policy.state_to_vector_fn(state),
            dtype=torch.float32
        )

        _, value = self.policy.net(x)
        return value

## Training function 

In [1533]:
def training(
    env:            Env,
    agent:          Agent_PPO,
    episodes:       int = 100,
    steps:          int = 100,
    rollout_length: int = 37,
    random_start:   bool = False
):
    # in training(), replace free_states with weighted sampling # we have an issue with the trap state that should be corrected and then generalized
    free_states    = [s for s in env.states if s.symbol not in ("X","O")]
    trap_neighbours = [
        s for s in free_states
        if any(n and n.symbol == "X" for n in s.neighbours.values())
    ]

    # weighted start: oversample trap neighbours so agent learns to avoid them
    #start_pool = free_states + trap_neighbours * 3  # 3x more likely to start near trap
    
    start_pool = env.states

    state_to_vector_fn = agent.policy.state_to_vector_fn

    # Episode loop

    for episode in range(episodes):

        if random_start:
            env.current = random.choice(start_pool)  # ← only picks from valid starts
        else:
            env.reset()

        s = env.current

        # guard: if somehow start is still terminal, skip episode
        if s.symbol in ("X", "O"):
            with torch.enable_grad():
                x     = torch.as_tensor(state_to_vector_fn(s),dtype=torch.float32)
                _,v_s = agent.policy.net(x)
                s.reward_()
                #r = torch.tensor(float(s.reward)) # target
                r = torch.tensor(float(s.reward)/100.0 , dtype= torch.float32) # target
                loss =F.mse_loss(v_s,r)
                agent.strategy.optimizer.zero_grad()
                loss.backward()
                agent.strategy.optimizer.step()
            
            continue

        # --- rollout buffer ---
        deltas         = []
        log_probs_old  = []
        states_visited = []
        actions_taken  = []
        values_visited = []
        dones = [] # to fix the leaking


        for step in range(steps):

            # --- agent acts ---
            a, log_prob, v_s = agent.act(s)

            # --- environment responds ---
            exp  = env.step(a)
            n_s  = exp["next_state"]
            r    = exp["reward"]
            done = exp["done"]

            # --- TD residual ---
            with torch.no_grad():
                x       = torch.as_tensor(state_to_vector_fn(n_s), dtype=torch.float32)
                _, v_ns = agent.policy.net(x)
                v_ns    = v_ns.squeeze() # newly added to garantee scalar

            # correct terminal bootstrap
            r = r / 100
            delta = (r - v_s.detach()) if done else \
                    (r + agent.strategy.gamma * v_ns - v_s.detach()) 

            # --- store ---
            deltas.append(delta)
            states_visited.append(s)
            actions_taken.append(a)
            values_visited.append(v_s.detach())
            log_probs_old.append(log_prob.detach())
            dones.append(done) # to store the dones to stop the leaking 

            s = n_s

            # --- PPO update every rollout_length steps ---


            if (step + 1) % rollout_length == 0:
                
                agent.learn({
                    "deltas":         deltas,
                    "states_visited": states_visited,
                    "log_probs_old":  log_probs_old,
                    "actions_taken":  actions_taken,
                    "values_visited": values_visited,
                    "dones": dones # to fix the leaking
                })
                deltas.clear(); log_probs_old.clear()
                states_visited.clear(); actions_taken.clear()
                values_visited.clear() ; dones.clear()

            # --- episode ended ---
            if done:

                break

        # --- learn whatever is left ---
        if len(deltas) > 0:
            agent.learn({
                "deltas":         deltas,
                "states_visited": states_visited,
                "log_probs_old":  log_probs_old,
                "actions_taken":  actions_taken,
                "values_visited": values_visited,
                "dones": dones
            })

In [1534]:
# ── usage ────────────────────────────────────────────────────────────────
n_features = len(state_to_vector(env.states[0]))   # 6
n_actions  = len(Actions)                           # 4

net       = ActorCritic(n_features=n_features, n_actions=n_actions, hidden_dim=64)
pi        = Policy_PPO(net ,Actions)
optimizer = torch.optim.Adam(net.parameters(), lr=3e-4)
ppo       = PPO(optimizer=optimizer, clip_epsilon=0.2, gamma=0.9, lam=0.95 ,cv = 0.5 , ce= 0.01 , n_epochs=4 ) # lam switch from 0.95 to 0.80
agent     = Agent_PPO(strategy=ppo, policy=pi)

env = Chessboard(board=claude_board, actions=Actions, start=s1)

training(env=env, agent=agent, episodes=5000 , steps = 200 ,rollout_length= 64 , random_start=True)

#debug_ppo(agent = agent,env = env)

/tmp/ipykernel_2279757/1169214734.py:50: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  if torch.isfinite(A).all() and A.std() > 1e-8: #I added the torch.isfinite to solve the NaN issue


In [1535]:
debug_ppo(agent = agent,env = env)
debug_learning_balance(agent.policy)


PPO COMPLETE DEBUG

[1] ENVIRONMENT
----------------------------------------------------------------------
 s1 -- up     --> s4  reward= -10.0 done=False symbol=None
 s1 -- right  --> s2  reward= -10.0 done=False symbol=None
 s1 -- down   --> s1  reward= -10.0 done=False symbol=None
 s1 -- left   --> s1  reward= -10.0 done=False symbol=None
 s2 -- up     --> s5  reward=-100.0 done=True symbol=X
 s2 -- right  --> s3  reward= -10.0 done=False symbol=None
 s2 -- down   --> s2  reward= -10.0 done=False symbol=None
 s2 -- left   --> s1  reward= -10.0 done=False symbol=None
 s3 -- up     --> s6  reward= -10.0 done=False symbol=None
 s3 -- right  --> s3  reward= -10.0 done=False symbol=None
 s3 -- down   --> s3  reward= -10.0 done=False symbol=None
 s3 -- left   --> s2  reward= -10.0 done=False symbol=None
 s4 -- up     --> s7  reward= -10.0 done=False symbol=None
 s4 -- right  --> s5  reward=-100.0 done=True symbol=X
 s4 -- down   --> s1  reward= -10.0 done=False symbol=None
 s4 -- left   -

/tmp/ipykernel_2279757/275698226.py:132: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.data.std().item():.6f}")
/tmp/ipykernel_2279757/275698226.py:154: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.grad.std().item():.6f}")


{'actor': 0.045455667013592986,
 'critic': 0.06219212241243886,
 'ratio': 0.7308909543757907}

In [1536]:
#testing the agent through episodes of multiples steps

for episode in range(10):
    print(f"in the episode {episode + 1} : \n")
    env.reset()
    for step in range(10-1):
        s = env.current
        a,log_pi_a_of_s ,v = agent.act(s)
        print(f"the action chosen for {s.name} is {a} \n")
        print(f"the value of that state is {v} \n")

        exp = env.step(a)

        if exp["done"] : 
            if exp["next_state"].symbol == "X" : symbol = "trap" 
            else: symbol = "goal"
            print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
            break

in the episode 1 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.5975569486618042 

the action chosen for s7 is right 

the value of that state is 0.79655921459198 

the action chosen for s8 is right 

the value of that state is 1.0161232948303223 

 we reached the state s9 which is the goal 

in the episode 2 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.5975569486618042 

the action chosen for s7 is right 

the value of that state is 0.79655921459198 

the action chosen for s8 is right 

the value of that state is 1.0161232948303223 

 we reached the state s9 which is the goal 

in the episode 3 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.5975569486618042 

the action chosen for s7 is r

In [1537]:

# also testing bu through all the possible boxes as start point
for start_state in env.states:

    for episode in range(10):
        
        env.current = start_state
        print(f"in the episode {episode + 1} for start {env.current.name} : \n")
        for step in range(10-1):
            s = env.current
            a,log_pi_a_of_s ,v = agent.act(s)
            print(f"the action chosen for {s.name} is {a} \n")
            print(f"the value of that state is {v} \n")

            exp = env.step(a)

            if exp["done"] : 
                if exp["next_state"].symbol == "X" : symbol = "trap" 
                else: symbol = "goal"
                print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
                break

in the episode 1 for start s1 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.5975569486618042 

the action chosen for s7 is right 

the value of that state is 0.79655921459198 

the action chosen for s8 is right 

the value of that state is 1.0161232948303223 

 we reached the state s9 which is the goal 

in the episode 2 for start s1 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.5975569486618042 

the action chosen for s7 is right 

the value of that state is 0.79655921459198 

the action chosen for s8 is right 

the value of that state is 1.0161232948303223 

 we reached the state s9 which is the goal 

in the episode 3 for start s1 : 

the action chosen for s1 is up 

the value of that state is 0.4542815685272217 

the action chosen for s4 is up 

the value of that state is 0.597556948

In [1538]:
print("=== Value estimates after training ===")
for s in env.states:
    with torch.no_grad():
        x = torch.as_tensor(state_to_vector(s), dtype=torch.float32)
        _, v = agent.policy.net(x)
    print(f"  V({s.name}) = {100*v.item():.3f}  symbol={s.symbol}")
    


=== Value estimates after training ===
  V(s1) = 45.428  symbol=None
  V(s2) = 63.263  symbol=None
  V(s3) = 82.526  symbol=None
  V(s4) = 59.756  symbol=None
  V(s5) = -99.033  symbol=X
  V(s6) = 99.671  symbol=None
  V(s7) = 79.656  symbol=None
  V(s8) = 101.612  symbol=None
  V(s9) = 100.799  symbol=O


In [1539]:
# distribution of actions 
print(Actions)
for s in States2:
    probs  = agent.policy.action_distribution(s)
    print(f"for the state {s.name} this is the action for each action: {100*probs}")

['up', 'right', 'down', 'left']
for the state s1 this is the action for each action: tensor([9.8266e+01, 1.0456e+00, 5.9341e-01, 9.5154e-02])
for the state s2 this is the action for each action: tensor([4.5018e-01, 9.9275e+01, 2.2765e-01, 4.7382e-02])
for the state s3 this is the action for each action: tensor([95.5759,  1.9741,  2.3246,  0.1254])
for the state s4 this is the action for each action: tensor([9.8988e+01, 3.8238e-01, 5.3935e-01, 9.0014e-02])
for the state s5 this is the action for each action: tensor([ 5.6858, 80.0235, 13.3945,  0.8963])
for the state s6 this is the action for each action: tensor([96.4619,  1.7223,  1.7173,  0.0985])
for the state s7 this is the action for each action: tensor([5.0327e-02, 9.9325e+01, 5.5642e-01, 6.8310e-02])
for the state s8 this is the action for each action: tensor([1.1286e-01, 9.9063e+01, 7.2966e-01, 9.4236e-02])
for the state s9 this is the action for each action: tensor([60.1955, 36.0594,  3.4865,  0.2586])


In [1540]:
# in training(), replace free_states with weighted sampling # we have an issue with the trap state that should be corrected and then generalized
free_states    = [s for s in env.states if s.symbol not in ("X","O")]
trap_neighbours = [
    s for s in free_states
    if any(n and n.symbol == "X" for n in s.neighbours.values())
]
print("=" * 40)

for s in trap_neighbours:
    env.current = s

    for a in Actions:

        if s.neighbours[a] == s5:

            exp = env.step(a)

            print(f"Start : {s.name}")
            print(f"Action: {a}")
            print(f"Next  : {exp['next_state'].name}")
            print(f"Reward: {exp['reward']}")
            print(f"Done  : {exp['done']}")
            print()

Start : s2
Action: up
Next  : s5
Reward: -100.0
Done  : True

Start : s4
Action: right
Next  : s5
Reward: -100.0
Done  : True

Start : s6
Action: left
Next  : s5
Reward: -100.0
Done  : True

Start : s8
Action: down
Next  : s5
Reward: -100.0
Done  : True



## testing in an other and bigger env

In [1541]:
b11 = CellBox("b11",(0,0),"white",None,[None,None,None,None])
b12 = CellBox("b12",(0,1),"white",None,[None,None,None,None])
b13 = CellBox("b13",(0,2),"white",None,[None,None,None,None])
b14 = CellBox("b14",(1,0),"white",None,[None,None,None,None])
b15 = CellBox("b15",(1,1),"white",None,[None,None,None,None])
b16 = CellBox("b16",(1,2),"white",None,[None,None,None,None])
b17 = CellBox("b17",(2,0),"white",None,[None,None,None,None])
b18 = CellBox("b18",(2,1),"white",None,[None,None,None,None])
b19 = CellBox("b19",(2,2),"white",None,[None,None,None,None])

Block1 = [b11, b12 , b13 , b14 ,b15 , b16 , b17 , b18 , b19 ]
Board1 = Board(Block1 , geometry= "box", data= {"shape":(3,3)})

b21 = CellBox("b21",(3,0),"white",None,[None,None,None,None])
b22 = CellBox("b22",(3,1),"white",None,[None,None,None,None])
b23 = CellBox("b23",(3,2),"white",None,[None,None,None,None])
b24 = CellBox("b24",(4,0),"white",None,[None,None,None,None])
b25 = CellBox("b25",(4,1),"white",None,[None,None,None,None])
b26 = CellBox("b26",(4,2),"white",None,[None,None,None,None])
b27 = CellBox("b27",(5,0),"white",None,[None,None,None,None])
b28 = CellBox("b28",(5,1),"white",None,[None,None,None,None])
b29 = CellBox("b29",(5,2),"white",None,[None,None,None,None])

Block2 = [b21, b22 , b23 , b24 ,b25 , b26 , b27 , b28 , b29 ]
Board2 = Board(Block2 , geometry= "box", data= {"shape":(3,3)})

b31 = CellBox("b31",(0,3),"white",None,[None,None,None,None])
b32 = CellBox("b32",(0,4),"white",None,[None,None,None,None])
b33 = CellBox("b33",(0,5),"white",None,[None,None,None,None])
b34 = CellBox("b34",(1,3),"white",None,[None,None,None,None])
b35 = CellBox("b35",(1,4),"white",None,[None,None,None,None])
b36 = CellBox("b36",(1,5),"white",None,[None,None,None,None])
b37 = CellBox("b37",(2,3),"white",None,[None,None,None,None])
b38 = CellBox("b38",(2,4),"white",None,[None,None,None,None])
b39 = CellBox("b39",(2,5),"white",None,[None,None,None,None])

Block3 = [b31, b32 , b33 , b34 ,b35 , b36 , b37 , b38 , b39 ]
Board3 = Board(Block3 , geometry= "box", data= {"shape":(3,3)})

b41 = CellBox("b41",(3,3),"white",None,[None,None,None,None])
b42 = CellBox("b42",(3,4),"white",None,[None,None,None,None])
b43 = CellBox("b43",(3,5),"white",None,[None,None,None,None])
b44 = CellBox("b44",(4,3),"white",None,[None,None,None,None])
b45 = CellBox("b45",(4,4),"white",None,[None,None,None,None])
b46 = CellBox("b46",(4,5),"white",None,[None,None,None,None])
b47 = CellBox("b47",(5,3),"white",None,[None,None,None,None])
b48 = CellBox("b48",(5,4),"white",None,[None,None,None,None])
b49 = CellBox("b49",(5,5),"white",None,[None,None,None,None])

Block4 = [b41, b42 , b43 , b44 ,b45 , b46 , b47 , b48 , b49 ]
Board4 = Board(Block4 , geometry= "box", data= {"shape":(3,3)})

b51 = CellBox("b51",(0,6),"white",None,[None,None,None,None])
b52 = CellBox("b52",(0,7),"white",None,[None,None,None,None])
b53 = CellBox("b53",(0,8),"white",None,[None,None,None,None])
b54 = CellBox("b54",(1,6),"white",None,[None,None,None,None])
b55 = CellBox("b55",(1,7),"white",None,[None,None,None,None])
b56 = CellBox("b56",(1,8),"white",None,[None,None,None,None])
b57 = CellBox("b57",(2,6),"white",None,[None,None,None,None])
b58 = CellBox("b58",(2,7),"white",None,[None,None,None,None])
b59 = CellBox("b59",(2,8),"white",None,[None,None,None,None])

Block5 = [b51, b52 , b53 , b54 ,b55 , b56 , b57 , b58 , b59 ]
Board5 = Board(Block5 , geometry= "box", data= {"shape":(3,3)})

b61 = CellBox("b61",(3,6),"white",None,[None,None,None,None])
b62 = CellBox("b62",(3,7),"white",None,[None,None,None,None])
b63 = CellBox("b63",(3,8),"white",None,[None,None,None,None])
b64 = CellBox("b64",(4,6),"white",None,[None,None,None,None])
b65 = CellBox("b65",(4,7),"white",None,[None,None,None,None])
b66 = CellBox("b66",(4,8),"white",None,[None,None,None,None])
b67 = CellBox("b67",(5,6),"white",None,[None,None,None,None])
b68 = CellBox("b68",(5,7),"white",None,[None,None,None,None])
b69 = CellBox("b69",(5,8),"white",None,[None,None,None,None])

Block6 = [b61, b62 , b63 , b64 ,b65 , b66 , b67 , b68 , b69 ]
Board6 = Board(Block6 , geometry= "box", data= {"shape":(3,3)})


# glew the boards together
Global_board = Board(Block1 + Block2 +Block3 + Block4 + Block5 +Block6 , geometry= "custom" ,
                      data= { "connections" :
                             [
                             (b17,"up",b21),(b18,"up",b22),(b19,"up",b23),
                             (b13,"right", b31), (b16,"right", b34), (b19,"right", b37),
                             (b23,"right", b41), (b26,"right", b44), (b29,"right", b47),
                             (b37 ,"up",b41), (b38 ,"up",b42), (b39,"up", b43 ),
                             (b43,"right", b61 ), (b46,"right", b64 ), (b49,"right", b67 ),
                             (b33,"right", b51 ), (b36,"right", b54 ), (b39,"right", b57 ),
                             (b61,"down", b57) , (b62,"down", b58) , (b63,"down", b59),
                             ]
                      }
                    )

# choose randomly which cells are traps and 3 goal points:

Blocks = Block1 + Block2 + Block3 + Block4 + Block5 + Block6
goals = 3
goal_cells = []
traps = 10 
trap_cells = []
normal_cells = []
for b in Blocks :
       symbol = random.choice(("X","O","_"))
       if symbol == "O" :
              if goals == 0 : 
                     normal_cells.append(b) 
                     continue

              goal_cells.append(b)
              goals -= 1
       if symbol == "X" :
              if traps == 0 : 
                     normal_cells.append(b) 
                     continue
              
              trap_cells.append(b)
              traps -= 1
       if symbol == "_":
              
              normal_cells.append(b) 
              continue

       b.symbol = symbol
       


In [1542]:
print("cells present in the goal list are: ")
for cell in goal_cells:
    print(f"the cell {cell.name}")

print("\n")

print("cells present in the trap list are")
for cell in trap_cells:
    print(f"the cell {cell.name}")

print("\n")

print("cells present in the normal list are")
for cell in normal_cells:
    print(f"the cell {cell.name}")

print("\n")

cells present in the goal list are: 
the cell b15
the cell b17
the cell b19


cells present in the trap list are
the cell b11
the cell b12
the cell b14
the cell b21
the cell b23
the cell b25
the cell b27
the cell b29
the cell b31
the cell b41


cells present in the normal list are
the cell b13
the cell b16
the cell b18
the cell b22
the cell b24
the cell b26
the cell b28
the cell b32
the cell b33
the cell b34
the cell b35
the cell b36
the cell b37
the cell b38
the cell b39
the cell b42
the cell b43
the cell b44
the cell b45
the cell b46
the cell b47
the cell b48
the cell b49
the cell b51
the cell b52
the cell b53
the cell b54
the cell b55
the cell b56
the cell b57
the cell b58
the cell b59
the cell b61
the cell b62
the cell b63
the cell b64
the cell b65
the cell b66
the cell b67
the cell b68
the cell b69




In [1548]:
env1 = Chessboard(board =Global_board , actions= Actions , start= b38 )

n_features1 = len(state_to_vector(env1.states[0]))
n_actions  = len(Actions)

net1 = ActorCritic(n_features1 , n_actions , hidden_dim=64 , n_layers= 4 )

optimizer = torch.optim.Adam(net1.parameters(), lr=3e-4)
ppo1 = PPO(optimizer= optimizer , clip_epsilon= 0.2 , cv = 0.25 , ce = 0.01 , gamma= 0.99 , lam = 0.95 ,n_epochs= 10 ) # epsilon = 0.2 , cv = 0.5

Pi1 = Policy_PPO(net1,env1.actions)

agent_carter = Agent_PPO(ppo1,Pi1)

training(env= env1 , agent= agent_carter , episodes= 10000 ,steps= 5000 , rollout_length= 64*2 , random_start=True)

/tmp/ipykernel_2279757/1169214734.py:50: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  if torch.isfinite(A).all() and A.std() > 1e-8: #I added the torch.isfinite to solve the NaN issue


In [1549]:
debug_learning_balance(agent_carter.policy)


LEARNING BALANCE
Actor gradient : 0.262695
Critic gradient: 0.382749
Ratio A/C      : 0.686
Status : Critic dominates


{'actor': 0.2626952936818263,
 'critic': 0.3827488782832874,
 'ratio': 0.6863384890811107}

In [1550]:
debug_ppo(agent = agent_carter ,env = env1)


PPO COMPLETE DEBUG

[1] ENVIRONMENT
----------------------------------------------------------------------
b11 -- up     --> b14 reward=-100.0 done=True symbol=X
b11 -- right  --> b12 reward=-100.0 done=True symbol=X
b11 -- down   --> b11 reward=-100.0 done=True symbol=X
b11 -- left   --> b11 reward=-100.0 done=True symbol=X
b12 -- up     --> b15 reward= 100.0 done=True symbol=O
b12 -- right  --> b13 reward= -10.0 done=False symbol=None
b12 -- down   --> b12 reward=-100.0 done=True symbol=X
b12 -- left   --> b11 reward=-100.0 done=True symbol=X
b13 -- up     --> b16 reward= -10.0 done=False symbol=None
b13 -- right  --> b31 reward=-100.0 done=True symbol=X
b13 -- down   --> b13 reward= -10.0 done=False symbol=None
b13 -- left   --> b12 reward=-100.0 done=True symbol=X
b14 -- up     --> b17 reward= 100.0 done=True symbol=O
b14 -- right  --> b15 reward= 100.0 done=True symbol=O
b14 -- down   --> b11 reward=-100.0 done=True symbol=X
b14 -- left   --> b14 reward=-100.0 done=True symbol=X


/tmp/ipykernel_2279757/275698226.py:132: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.data.std().item():.6f}")
/tmp/ipykernel_2279757/275698226.py:154: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1861.)
  print(f" std  : {p.grad.std().item():.6f}")


In [1552]:
## testing the agent through episodes of multiples steps

for episode in range(10):
    print(f"in the episode {episode + 1} : \n")
    env1.reset()
    for step in range(500-1):
        s = env1.current
        a,log_pi_a_of_s ,v = agent.act(s)
        print(f"the action chosen for {s.name} is {a} \n")
        print(f"the value of that state is {v} \n")

        exp = env1.step(a)

        if exp["done"] : 
            if exp["next_state"].symbol == "X" : symbol = "trap" 
            else: symbol = "goal"
            print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
## also testing bu through all the possible boxes as start point
# for start_state in env.states:

#     for episode in range(10):
        
#         env.current = start_state
#         print(f"in the episode {episode + 1} for start {env.current.name} : \n")
#         for step in range(10-1):
#             s = env.current
#             a,log_pi_a_of_s ,v = agent.act(s)
#             print(f"the action chosen for {s.name} is {a} \n")
#             print(f"the value of that state is {v} \n")

#             exp = env.step(a)

#             if exp["done"] : 
#                 if exp["next_state"].symbol == "X" : symbol = "trap" 
#                 else: symbol = "goal"
#                 print(f" we reached the state {exp["next_state"].name} which is the {symbol} \n")
#                 break

in the episode 1 : 

the action chosen for b38 is up 

the value of that state is -0.5945550203323364 

the action chosen for b42 is up 

the value of that state is 0.11254161596298218 

the action chosen for b45 is up 

the value of that state is -0.5945550203323364 

the action chosen for b48 is right 

the value of that state is 0.9034164547920227 

the action chosen for b49 is right 

the value of that state is 0.9034164547920227 

the action chosen for b67 is right 

the value of that state is 0.9034164547920227 

the action chosen for b68 is right 

the value of that state is 0.9034164547920227 

the action chosen for b69 is up 

the value of that state is 1.0287303924560547 

the action chosen for b69 is up 

the value of that state is 1.0287303924560547 

the action chosen for b69 is up 

the value of that state is 1.0287303924560547 

the action chosen for b69 is up 

the value of that state is 1.0287303924560547 

the action chosen for b69 is up 

the value of that state is 1

### Potential issue

The only thing to be careful about is that everyone must reference the same network object.

#### Good:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())

ppo = PPO(network=net, optimizer=optimizer)
```

All three share the same net.

#### Bad:
```python
policy = Policy_PPO(ActorCritic(...))
ppo = PPO(network=ActorCritic(...), ...)
``` 
Now there are two different networks:

policy uses one,
PPO updates another.

The policy will never see the learned weights.

### Recommendation

Pass the same network instance everywhere:
```python
net = ActorCritic(...)

policy = Policy_PPO(net)
optimizer = torch.optim.Adam(net.parameters())
strategy = PPO(net, optimizer)
agent = Agent(strategy, policy)
```
This architecture is clean, modular, and very close to how RL libraries (e.g. Stable-Baselines3, CleanRL) organize actor-critic agents.

In [ ]:
# import the sb3 here and see how to integrate them with my code (see if i can package it under my designed classes)


In [ ]:
# import the gymnasium stuff here